Multi-Model Integration: Using Gemini, DeepSeek & Groq with OpenAI Agents

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel
from openai.types.responses import ResponseTextDeltaEvent
import asyncio
from openai import AsyncOpenAI
import os
import sendgrid
from sendgrid.helpers.mail import Email, To, Content, Mail
#Load environment variable
load_dotenv(override=True)

True

In [2]:
#Load API keys and verify
api_key = {
    "deepseek" : os.getenv("API_TOKEN"),
    "Gemini" : os.getenv("GEMINI_API_TOKEN")
}

for provider, key in api_key.items():
    if key:
        print(f"{provider.title()} API Key loaded: {key[:5]}")
    else:
        print(f"{provider.title} API key is missing.")

Deepseek API Key loaded: sk-2d
Gemini API Key loaded: AQ.Ab


Define sales agent personas

In [3]:
# Personalized agent instructions
agent_styles = {
    "direct": "You write serious, clear, and professional sales emails for SOC2 compliance solutions.",
    "witty": "You write humorous, attention-grabbing emails for SOC2 compliance tools.",
    "concise": "You write minimal, punchy cold emails ideal for busy executives."
}

Initialize multi-model client

In [4]:
# Base URLs for LLMs
MODEL_BASE_URLS = {
    "Gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "deepseek": "https://api.deepseek.com/v1"
}
#Create Agent clients
model_clients = {
    "Gemini" : AsyncOpenAI(base_url=MODEL_BASE_URLS["Gemini"], api_key=api_key["Gemini"]),
    "deepseek" : AsyncOpenAI(base_url=MODEL_BASE_URLS["deepseek"], api_key=api_key["deepseek"])
}


Build model specific agents

In [7]:
#Create the agent of LLMs
agent_models = {
    "Gemini" : OpenAIChatCompletionsModel("gemini-2.0-flash", model_clients["Gemini"]),
    "deepseek" : OpenAIChatCompletionsModel("deepseek-flash", model_clients["deepseek"])
}
# Sales Agents - Multi LLM Model Based
sales_agents = [
    Agent(name="Gemini Agent", instructions=agent_styles["witty"], model=agent_models['Gemini']),
    Agent(name="DeepSeek Agent", instructions=agent_styles["direct"], model=agent_models["deepseek"])
]
# Convert agents to tools
email_task_description = "Generate a cold outreach email for a SaaS compliance product."
agent_tools = [
    agent.as_tool(tool_name=f"{agent.name.lower().replace(' ', '_')}_tool", tool_description=email_task_description)
    for agent in sales_agents
]

agent_tools


[FunctionTool(name='gemini_agent_tool', description='Generate a cold outreach email for a SaaS compliance product.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x10f8be8a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None),
 FunctionTool(name='deepseek_agent_tool', description='Generate a cold outreach email for a SaaS compliance product.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Inp

Email Formatting & Sending Utilities

Tool: SendGrid Email Sender

In [8]:
# Function Tool to Send Email via SendGrid
@function_tool
def send_email(subject: str, html_body: str):
     """Send an HTML email with subject and body to the recipient list."""
     sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
     sender = Email("anshulc55@gmail.com")
     recipient = To("anshulc55@icloud.com")
     content = Content("text/html", html_body)
     email = Mail(sender, recipient, subject, content).get()
     response = sg.client.mail.send.post(request_body=email)
     return {"status": "Success" , "code": response.status_code}

Utility Agents

In [11]:
# Subject Line Generator
subject_agent = Agent(
    name="Subject Line Creator",
    instructions="Write catchy subject lines for cold outreach emails.",
    model="deepseek-flash"
)
subject_tool = subject_agent.as_tool("subject_writer", "Generate a catchy email subject line")

# HTML Formatter
html_agent = Agent(
    name="HTML Formatter",
    instructions="Convert plain text emails (with markdown) to clean HTML layouts.",
    model="deepseek-flash"
)
html_tool = html_agent.as_tool("html_converter", "Convert plain text email to HTML")

Email Orchestration Agent

In [12]:
email_utilities = [subject_tool, html_tool, send_email]

email_formatter = Agent(
    name="Email Orchestrator",
    instructions="Take a plain email body, generate a subject, convert to HTML, and send it.",
    tools=email_utilities,
    model="deepseek-flash",
    handoff_description="Format and send the final sales email"
)

Sales Manager Agent: Multi-Model Integration

In [13]:
manager_instructions = """You are a sales manager for TechLink. Your job is to:
1. Try all available sales agents at least once.
2. Evaluate the quality of their emails.
3. Choose the most effective email.
4. Hand off to the Email Orchestrator agent for formatting and sending."""

sales_manager = Agent(
    name="Sales Manager",
    instructions=manager_instructions,
    tools=agent_tools,
    handoffs=[email_formatter],
    model="deepseek-flash"
)

Execution and Tracing

In [15]:
message = "Send out a cold sales email addressed to Dear CEO from Alice"
with trace("Multi-Model Sales Email Automation"):
    result = await Runner.run(sales_manager, message)

Error getting response


NotFoundError: Error code: 404 - {'error': {'message': 'The model `deepseek-flash` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

Example: Guardrail for Name Detection

In [16]:
#Prevent the sales agent from running if the input message includes a personal name (e.g., "Alice", "John").
# Define the Structured Output Schema
from pydantic import BaseModel

class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

## - Schema enforces that the output from the name-detection agent must include: is_name_in_message, name

In [19]:
# Create the Guardrail Agent
guardrail_agent = Agent(
    name="Name Validator",
    instructions="Analyze the input to determine if a personal name is present.",
    output_type=NameCheckOutput,
    model="deepseek-flash"
)

## - This agent uses the structured output to enforce strict control over its response format.

Define the Input Guardrail Function